## Preparing Data for a model to forecast the country's total energy consumption

As the title suggests, in this notebook, I intend to focus on preparing data for the model.

I am beginning the load forecasting task immediately after the successful implementation of an offshore wind power forecasting model, as I believe that features derived from the load data could be useful when developing more complex models later on.

In my current approach, I am operating on the premise that the load is, to a large extent, dependent on calendar-based features.

**However, the plan is to try incorporating features such as temperature or storm warnings into the set of predictors, as these can affect heating costs or, theoretically, lead to the suspension of operations and a drop in load.**

In [12]:
# Imports
import pandas as pd
import numpy as np
import os
import pyrootutils
import holidays
from dotenv import load_dotenv

# Needed to load variables from the .env file into the environment
load_dotenv()

# Determining the project root based on the .gitignore file
PROJECT_ROOT = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)
from utils.offshore_utils import fetch_offshore_weather

# Determining data directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw_data" / "load_model"
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "clean_data" / "load_model"
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets" / "load_model"

In [6]:
# Determining the time interval for the energy load data
START = '2026-01-01'
END = '2026-06-20'

# Creating a DataFrame with UTC and Berlin timestamps
# Index (UTC Timestamps)
utc_index = pd.date_range(
    start=START, 
    end=END, 
    freq='15min', 
    tz='UTC'
)

# Converting UTC timestamps to Berlin time
berlin_time = utc_index.tz_convert('Europe/Berlin')

# Combining both UTC and Berlin timestamps into a single DataFrame
df = pd.DataFrame({
    'dt_berlin': berlin_time
}, index=utc_index)

# Displaying a sample of the DataFrame to verify its structure
df.sample(5)  # Displaying a sample of the DataFrame to verify its structure

,dt_berlin
2026-02-08 20:45:00+00:00,2026-02-08 21:45:00+01:00
2026-05-31 03:00:00+00:00,2026-05-31 05:00:00+02:00
2026-04-15 06:45:00+00:00,2026-04-15 08:45:00+02:00
2026-04-12 04:00:00+00:00,2026-04-12 06:00:00+02:00
2026-03-15 05:00:00+00:00,2026-03-15 06:00:00+01:00


In [21]:

# Adding a column for the quarter index based on Berlin time
from numpy import int8


minutes_from_midnight = df['dt_berlin'].dt.hour * 60 + df['dt_berlin'].dt.minute
quarter_index = minutes_from_midnight // 15

df['quarter_index'] = quarter_index

# Adding a column for the day of the week based on Berlin time
df['dow'] = df['dt_berlin'].dt.day_name()

# Adding a column to indicate whether the day is a weekday
df['is_weekday'] = (df['dt_berlin'].dt.weekday < 5).astype(int8)  # True for Monday to Friday, False for Saturday and Sunday

# Adding a column to indicate whether the day is a weekend
df['is_weekend'] = (df['dt_berlin'].dt.weekday >= 5).astype(int8)  # True for Saturday and Sunday, False for Monday to Friday

years = df['dt_berlin'].dt.year.unique()
df['is_holiday'] = df['dt_berlin'].dt.date.isin(holidays.Germany(years=years)).astype(int8)  # True for holidays, False otherwise

# Adding a column for the day of the year based on Berlin time
df['doy'] = df['dt_berlin'].dt.dayofyear

df.sample(5)

,dt_berlin,quarter_index,dow,doy,is_workday,is_weekday,is_weekend,is_holiday
2026-06-04 13:45:00+00:00,2026-06-04 15:45:00+02:00,63,Thursday,155,1,1,0,0
2026-04-13 06:15:00+00:00,2026-04-13 08:15:00+02:00,33,Monday,103,1,1,0,0
2026-04-07 23:30:00+00:00,2026-04-08 01:30:00+02:00,6,Wednesday,98,1,1,0,0
2026-02-27 13:30:00+00:00,2026-02-27 14:30:00+01:00,58,Friday,58,1,1,0,0
2026-03-18 17:15:00+00:00,2026-03-18 18:15:00+01:00,73,Wednesday,77,1,1,0,0


In [ ]:
# Information taken from:
# Monitoringbericht 2025 - Monitoringbericht gemäß § 63 Abs. 3 i. V. m. § 35 EnWG und § 48 Abs. 3 i. V. m. § 53 Abs. 3 GWB
# https://data.bundesnetzagentur.de/Bundesnetzagentur/SharedDocs/Mediathek/Monitoringberichte/MonitoringberichtEnergie2025.pdf
# Pages 96-97
# Consumption volumes (TWh) by states at the VNB level for 2024 (Bundesnetzagentur)
consumption_twh = {
    'NW': 86.94,  # Nordrhein-Westfalen
    'BY': 63.02,  # Bayern
    'BW': 52.09,  # Baden-Württemberg
    'HE': 40.45,  # Hessen
    'NI': 35.58,  # Niedersachsen
    'SN': 19.81,  # Sachsen
    'RP': 17.97,  # Rheinland-Pfalz
    'ST': 13.48,  # Sachsen-Anhalt
    'BE': 12.31,  # Berlin
    'BB': 11.59,  # Brandenburg
    'SH': 11.22,  # Schleswig-Holstein
    'TH': 10.05,  # Thüringen
    'HH': 9.91,   # Hamburg
    'MV': 6.06,   # Mecklenburg-Vorpommern
    'SL': 5.07,   # Saarland
    'HB': 4.47    # Bremen
}

# Normalizing the consumption volumes to get weights for each state
total_twh = sum(consumption_twh.values())
state_weights = {state: val / total_twh for state, val in consumption_twh.items()}

state_weights

{'NW': 0.21733913304334784,
 'BY': 0.15754212289385533,
 'BW': 0.13021848907554623,
 'HE': 0.10111994400279987,
 'NI': 0.08894555272236389,
 'SN': 0.04952252387380631,
 'RP': 0.04492275386230688,
 'ST': 0.03369831508424579,
 'BE': 0.030773461326933656,
 'BB': 0.02897355132243388,
 'SH': 0.028048597570121497,
 'TH': 0.025123743812809362,
 'HH': 0.024773761311934404,
 'MV': 0.015149242537873106,
 'SL': 0.012674366281685917,
 'HB': 0.011174441277936103}